In [24]:
#!pip install xgboost scikit-learn joblib pandas numpy fastapi uvicorn pyngrok

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [26]:
file_patch = 'Laptop_price.csv'
df = pd.read_csv(file_patch)
X = df.drop(columns=['Price'])
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import joblib

In [28]:
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, num_features)
])
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5))
])
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, 'Laptop_price_model.pkl')

['Laptop_price_model.pkl']

In [29]:
!git init

Reinitialized existing Git repository in C:/Users/kosta/OneDrive/Рабочий стол/Питон/4 сем/.git/


In [31]:
!git fetch
!git branch -a

* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main


In [32]:
!git add lab1_Laptev_pipeline.ipynb
!git add Laptop_price.csv

In [33]:
!git commit -m"Добавлен ML-пайплайн"

[main d285341] Добавлен ML-пайплайн
 1 file changed, 45 insertions(+), 48 deletions(-)


In [34]:
!git remote add origin https://github.com/BorgTheHead/1lab_pipeline.git


error: remote origin already exists.


In [35]:
!git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/BorgTheHead/1lab_pipeline.git
   e07e019..d285341  main -> main


In [36]:
%%writefile app.py
from fastapi import FastAPI, File, UploadFile
import pandas as pd
import joblib
from io import BytesIO

app = FastAPI()

model_patch = "laptop_price_model.pkl"
model = joblib.load(model_patch)

@app.post("/predict/")
async def predict(file: UploadFile = File(...)):
    content = await file.read()
    df = pd.read_csv(BytesIO(content))
    predictions = model.predict(df)
    return {"predictions": predictions.tolist()}

Overwriting app.py


In [37]:
!ngrok config add-authtoken 2wTAwLnIdlKLFVxMMWpIN2eX4zE_2ZNthzYkBqnnuQMLdCY2Z

                                                                                                    
Installing ngrok ... 
                                                                                                    
Authtoken saved to configuration file: C:\Users\kosta\AppData\Local/ngrok/ngrok.yml


In [40]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 --reload > fastapi.log 2>&1 &

OSError: Background processes not supported.

In [41]:
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print("API доступно по адресу:", public_url)


t=2025-05-01T02:31:43+0300 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: We do not allow agents to connect to ngrok from your IP address (109.63.226.248).\r\n\r\nERR_NGROK_9040\r\n"
t=2025-05-01T02:31:43+0300 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: We do not allow agents to connect to ngrok from your IP address (109.63.226.248).\r\n\r\nERR_NGROK_9040\r\n"
t=2025-05-01T02:31:43+0300 lvl=eror msg="terminating with error" obj=app err="authentication failed: We do not allow agents to connect to ngrok from your IP address (109.63.226.248).\r\n\r\nERR_NGROK_9040\r\n"
t=2025-05-01T02:31:43+0300 lvl=crit msg="command failed" err="authentication failed: We do not allow agents to connect to ngrok from your IP address (109.63.226.248).\r\n\r\nERR_NGROK_9040\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: We do not allow agents to connect to ngrok from your IP address (109.63.226.248).\r\n\r\nERR_NGROK_9040\r\n.